In [1]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_huggingface import HuggingFaceEmbeddings
import os

d:\scaleRAG\productionRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_17968\3169841072.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
def rewrite_for_retrieval(query, chat_history=None):

    if not chat_history:
        return query

    history = "\n".join(
        f"{msg['role']}: {msg['content']}"
        for msg in chat_history[-4:]
    )

    prompt = f"""
    Conversation:
    {history}

    Rewrite the query so it is self-contained and
    optimized for semantic search.

    Query:
    {query}
    """

    return llm.invoke(prompt).content.strip()

In [ ]:
def hyde_search(query, k=3):

    prompt = f"""
    Write a short answer to the following question.

    Question:
    {query}
    """

    hypothetical_answer = llm.invoke(prompt).content

    docs = vectorstore.similarity_search(
        hypothetical_answer,
        k=k
    )

    return docs

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [5]:
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [6]:
index_name = "prod-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [7]:
file_path = "D:\ProdRAG\prodRAG\example.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [9]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2894.87it/s]


In [10]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [11]:
retriever= vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [12]:
# bm25 = BM25Retriever.from_documents(texts)
# bm25.k = 5

# dense = vectorstore.as_retriever(search_kwargs={"k": 5})

# retriever_2 = EnsembleRetriever(
# retrievers=[bm25, dense],
# weights=[0.4, 0.6]
# )

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
def chunks_docs(docs):
    return [doc.page_content for doc in docs]

In [15]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [16]:
# rag_chain_1 = RunnableParallel(
#     {
#         "question": RunnablePassthrough(),
#         "context": retriever_1 | RunnableLambda(format_docs),
        
#     }
# )

# rag_chain_2 = RunnableParallel(
#     {
#         "question": RunnablePassthrough(),
#         "context": retriever_2 | RunnableLambda(format_docs),
#     }
# )


In [17]:
# result_1=rag_chain_1 | rag_prompt | llm
# result_2=rag_chain_2 | rag_prompt | llm

In [18]:
# op_1=result_1.invoke("What is summary of module?")
# op_2=result_2.invoke("What is summary of module?")

In [19]:
# print(op_1.content)


In [ ]:
# print(op_2.content)

The context provides a summary of the course modules as follows:

- Module 2: Cryptography Fundamentals
- Module 3: Consensus Algorithms
- Module 4: Smart Contracts and Ethereum
- Module 5: Blockchain and AI Integration
- Module 7: Cybersecurity and Blockchain
- Module 8: Case Studies and Project Work

These modules cover various topics related to blockchain technology, including cryptography, consensus algorithms, smart contracts, AI integration, cybersecurity, and case studies.


In [ ]:
def ask_rag(query, chat_history=None):

    rewritten_query = rewrite_for_retrieval(
        query,
        chat_history
    )

    docs = hyde_search(
        rewritten_query
    )

    context = format_docs(docs)

    prompt = rag_prompt.format(
        context=context,
        question=query
    )

    response = llm.invoke(prompt)

    return response.content

In [ ]:
op = ask_rag(
    "What is summary of module?"
)



In [ ]:
print(op)